<a href="https://colab.research.google.com/github/shirsh008/Ropeway---IPT-/blob/main/04_ghsl_demand_scaleup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install "city2graph"
%pip install contextily
%pip install -U geovoronoi[plotting]
%pip install rasterio
%pip install pyproj

In [ ]:
import city2graph as c2g
import osmnx as ox
import geopandas as gpd
import geopandas.plotting
from shapely.geometry import Point, MultiPolygon
from pyproj import Transformer
def _flatten_multi_geoms(geoms, prefix=None):
    """
    Replacement for the missing GeoPandas function.
    Returns TWO lists: (flattened_geometries, original_indices)
    """
    flattened_geoms = []
    flattened_indices = []

    for i, geom in enumerate(geoms):
        if isinstance(geom, MultiPolygon):
            for g in geom.geoms:
                flattened_geoms.append(g)
                flattened_indices.append(i)
        else:
            flattened_geoms.append(geom)
            flattened_indices.append(i)

    return flattened_geoms, flattened_indices
geopandas.plotting._flatten_multi_geoms = _flatten_multi_geoms
import matplotlib.pyplot as plt
import contextily as cx
import matplotlib.lines as Line2D
import shapely.geometry as box
from shapely.geometry import LineString, Point
import rasterio as rio
import pandas as pd
import networkx as nx
import folium
from geovoronoi import voronoi_regions_from_coords, points_to_coords
from geovoronoi.plotting import subplot_for_map, plot_voronoi_polys_with_points_in_area
from sklearn.cluster import KMeans
import seaborn as sns
import time
import numpy as np
from rasterio.enums import Resampling
from folium.plugins import HeatMap
from tqdm import tqdm

In [ ]:
pop_raster = rio.open('final_new_pop_varanasi.tif')
work_raster = rio.open('work_area_varanasi.tif')
# x,y = pop_raster.xy(100,200)
pop_data = pop_raster.read(1)
work_data = work_raster.read(1)
pop_lon = []
pop_lat = []
pop = []
work = []
work_lon = []
work_lat = []
trnsformer = Transformer.from_crs("esri:54009", "epsg:4326", always_xy = True)
for i in range(len(pop_data)):
  for j in range(len(pop_data[i])):
    if(0 < pop_data[i][j] <= 100):
      pop.append(pop_data[i][j])
      x,y = pop_raster.xy(i,j)
      lon,lat = trnsformer.transform(x,y)
      pop_lon.append(lon)
      pop_lat.append(lat)

for i in range(len(work_data)):
  for j in range(len(work_data[i])):
    if(0 < work_data[i][j] <= 100):
      work.append(work_data[i][j])
      x,y = work_raster.xy(i,j)
      lon,lat = trnsformer.transform(x,y)
      work_lon.append(lon)
      work_lat.append(lat)


In [ ]:
city_name ="Varanasi"
G = ox.graph_from_place(city_name, network_type="drive")
admin = ox.geocode_to_gdf(city_name)

In [ ]:
G_projected = ox.project_graph(G)
admin_projected = admin.to_crs(G_projected.graph['crs'])

In [ ]:
fig, ax = ox.plot_graph(G, show=False, close=False, figsize=(10,10), bgcolor='white', edge_color='#999999', node_size=0)
plt.legend()
plt.show()

In [ ]:
a = pop_lat[0]
print(a, pop_lon[0])
print(len(pop_lat))

In [ ]:
pop_raster = rio.open('final_new_pop_varanasi.tif')
work_raster = rio.open('work_area_varanasi.tif')
x,y = pop_raster.xy(100,200)
pop_data = pop_raster.read(1)
work_data = work_raster.read(1)
pop_lon = []
pop_lat = []
work_lon = []
work_lat = []
trnsformer = Transformer.from_crs("ESRI:54009", "EPSG:4326", always_xy = True)
print(trnsformer.transform(x,y))

In [ ]:
from pyproj import Transformer

with rio.open('final_population_varanasi.tif') as src:

    step = 1
    band = src.read(1)[::step, ::step]

    height, width = band.shape
    cols, rows = np.meshgrid(np.arange(0, src.width, step), np.arange(0, src.height, step))

    xs, ys = rio.transform.xy(src.transform, rows, cols)

    transformer = Transformer.from_crs(src.crs, 'EPSG:4326', always_xy=True)
    lons, lats = transformer.transform(np.array(xs).flatten(), np.array(ys).flatten())
    values = band.flatten()
df = pd.DataFrame({'lon': lons, 'lat': lats, 'value': values})
df = df[df['value'] != src.nodata].dropna()

m = folium.Map(location=[df.lat.mean(), df.lon.mean()], zoom_start=12, tiles='OpenStreetMap')

for i, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        popup=f"Value: {row['value']}",
        color='blue',
        fill=True,
        fill_opacity=0.7
    ).add_to(m)

m

In [ ]:
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326"
)
gdf = gdf.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(12, 10))


gdf.plot(ax=ax, marker='o', c=gdf['value'], cmap='YlOrRd',
         markersize=10, alpha=0.7, legend=True,
         legend_kwds={'label': "Population Value"})

cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

ax.set_title("Population Distribution - Varanasi", fontsize=15)
ax.set_axis_off()

plt.show()

In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt

G_projected = ox.project_graph(G)

from pyproj import Transformer
target_crs = G_projected.graph['crs']
transformer = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)

proj_lon, proj_lat = transformer.transform(df['lon'].values, df['lat'].values)

fig, ax = ox.plot_graph(
    G_projected,
    show=False,
    close=False,
    figsize=(12, 12),
    bgcolor='white',
    edge_color='#999999',
    edge_linewidth=0.5,
    node_size=0
)

scatter = ax.scatter(
    proj_lon,
    proj_lat,
    c=df['value'],
    cmap='Blues',
    s=50,
    marker='.',
    zorder=5,
    label='Population Density'
)

plt.colorbar(scatter, ax=ax, label='Population Value', shrink=0.5)
ax.set_title("Varanasi Road Network & Population Overlay", fontsize=15)
plt.legend(loc='upper right')
plt.show()

In [ ]:
from pyproj import Transformer

with rio.open('final_work_area.tif') as src:

    step = 1
    band = src.read(1)[::step, ::step]

    height, width = band.shape
    cols, rows = np.meshgrid(np.arange(0, src.width, step), np.arange(0, src.height, step))

    xs, ys = rio.transform.xy(src.transform, rows, cols)

    transformer = Transformer.from_crs(src.crs, 'EPSG:4326', always_xy=True)
    lons, lats = transformer.transform(np.array(xs).flatten(), np.array(ys).flatten())
    values = band.flatten()


df_work = pd.DataFrame({'lon': lons, 'lat': lats, 'value': values})
df_work = df_work[df_work['value'] != src.nodata].dropna()

m = folium.Map(location=[df_work.lat.mean(), df_work.lon.mean()], zoom_start=12, tiles='OpenStreetMap')

for i, row in df_work.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        popup=f"Value: {row['value']}",
        color='blue',
        fill=True,
        fill_opacity=0.7
    ).add_to(m)

m

In [ ]:
print(len(df_work['lon']))
print(len(df['lon']))

In [ ]:
%pip install osmnx geopandas networkx shapely

In [ ]:
pop_lon = df['lon'].values
pop_lat = df['lat'].values

work_lon = df_work['lon'].values
work_lat = df_work['lat'].values

print("Snapping points to the nearest road nodes...")
pop_nodes = ox.distance.nearest_nodes(G, X=pop_lon, Y=pop_lat)
work_nodes = ox.distance.nearest_nodes(G, X=work_lon, Y=work_lat)

print("Reversing graph arrows for Many-to-Few routing...")
G_rev = G.reverse(copy=True)

print("Creating high-speed coordinate dictionaries...")
nodes, edges = ox.graph_to_gdfs(G)
node_x_dict = nodes['x'].to_dict()
node_y_dict = nodes['y'].to_dict()

routes_data = []

for w_idx, w_node in enumerate(tqdm(work_nodes, desc="Processing Work Hubs")):
    try:
        paths = nx.single_source_dijkstra_path(G_rev, source=w_node, weight='length')

        for p_idx, p_node in enumerate(pop_nodes):
            if p_node in paths:


                route_nodes = paths[p_node][::-1]

                route_coords = [(node_x_dict[node], node_y_dict[node]) for node in route_nodes]

                if len(route_coords) > 1:
                    route_geom = LineString(route_coords)

                    routes_data.append({
                        'pop_index': p_idx,
                        'work_index': w_idx,
                        'geometry': route_geom
                    })
    except Exception as e:
        pass


In [ ]:
import matplotlib.pyplot as plt

print("Google's servers are drawing the map. Please wait about 60 seconds...")

fig, ax = plt.subplots(figsize=(20, 20), facecolor='black')
ax.set_facecolor('black')
routes_gdf.plot(ax=ax, color='cyan', linewidth=0.2, alpha=0.02)

ax.axis('off')

plt.show()

In [ ]:
routes_gdf = gpd.GeoDataFrame(routes_data, geometry='geometry', crs = "EPSG:4326")

In [ ]:
print("Extracting base map geometries...")
nodes, edges = ox.graph_to_gdfs(G)

print("Setting up the canvas...")
fig, ax = plt.subplots(figsize=(12, 12), facecolor='white')
ax.set_facecolor('white')

print("Drawing the grey street network...")
edges.plot(ax=ax, linewidth=0.5, edgecolor='darkgrey', alpha=0.6)

print("Overlaying your calculated routes...")
routes_gdf.plot(ax=ax, color='blue', linewidth=0.2, alpha=0.03)

ax.axis('off')

print("Rendering the final image. This will take a moment...")
plt.show()

In [ ]:
routes_gdf_constrained = gpd.GeoDataFrame(routes_data, geometry='geometry', crs = "EPSG:4326")

In [ ]:
print("Applying road closures...")

to_remove_new = [8779060366, 8779060372, 293430853, 293434638, 292741298, 8779060374, 1464411802, 293435628, 1464411802, 8779060363, 8779060362, 8779060361, 8779060360, 8883310761, 8907128658, 8882692898]

valid_nodes_to_skip = [node for node in to_remove_new if node in G.nodes]

G_detour = G.copy()
G_detour.remove_nodes_from(valid_nodes_to_skip)
print(f"Successfully closed {len(valid_nodes_to_skip)} intersections out of {len(to_remove_new)} requested.")
print("Reversing graph arrows for Many-to-Few routing...")
G_rev = G_detour.reverse(copy=True)

print("Creating high-speed coordinate dictionaries...")
nodes, edges = ox.graph_to_gdfs(G)
node_x_dict = nodes['x'].to_dict()
node_y_dict = nodes['y'].to_dict()

routes_data_new = []

for w_idx, w_node in enumerate(tqdm(work_nodes, desc="Processing Work Hubs")):
    try:
        paths = nx.single_source_dijkstra_path(G_rev, source=w_node, weight='length')

        for p_idx, p_node in enumerate(pop_nodes):
            if p_node in paths:
                route_nodes = paths[p_node][::-1]

                route_coords = [(node_x_dict[node], node_y_dict[node]) for node in route_nodes]

                if len(route_coords) > 1:
                    route_geom = LineString(route_coords)

                    routes_data_new.append({
                        'pop_index': p_idx,
                        'work_index': w_idx,
                        'geometry': route_geom
                    })
    except Exception as e:
        pass

In [ ]:
print("Extracting base map geometries...")
nodes, edges = ox.graph_to_gdfs(G)

print("Setting up the canvas...")
fig, ax = plt.subplots(figsize=(12, 12), facecolor='white')
ax.set_facecolor('white')

print("Drawing the grey street network...")
edges.plot(ax=ax, linewidth=0.5, edgecolor='darkgrey', alpha=0.6)

print("Overlaying your CONSTRAINED routes...")
routes_gdf_constrained.plot(ax=ax, color='blue', linewidth=0.2, alpha=0.03)

ax.axis('off')

print("Rendering the final image. This will take a moment...")
plt.show()

In [ ]:
# The [:10] slice ensures the loop only runs for the first 10 routes
for route in routes_data_new[:10]:
    # Extract the coordinates from the LineString geometry
    coords = list(route['geometry'].coords)

    print(f"Path from Work {route['work_index']} to Pop {route['pop_index']}:")
    print(coords)
    print("-" * 20)

In [ ]:
unique_coords = set()
sum = 0
for route in routes_data_new:
    sum += len(route)
    coords = list(route['geometry'].coords)
    unique_coords.update(coords)

total_nodes_used = len(unique_coords)
print(f"Total unique nodes used in the map: {total_nodes_used}")
print(sum)

In [ ]:
print(len(work_nodes))
print(len(pop_lon))

In [ ]:
from collections import Counter

demand_map_new = Counter()

for p_node in pop_nodes:
    x = node_x_dict[p_node]
    y = node_y_dict[p_node]

    demand_map_new[(x, y)] = 1

for route in routes_data_new:
    coords = list(route['geometry'].coords)

    intersected_nodes = coords[1:]

    demand_map_new.update(intersected_nodes)

print(f"Demand computation complete! Tracked {len(demand_map_new)} unique nodes.")

In [ ]:
from collections import Counter

demand_map = Counter()

for p_node in pop_nodes:
    x = node_x_dict[p_node]
    y = node_y_dict[p_node]

    demand_map[(x, y)] = 1

for route in routes_data:
    coords = list(route['geometry'].coords)

    intersected_nodes = coords[1:]

    demand_map.update(intersected_nodes)

print(f"Demand computation complete! Tracked {len(demand_map)} unique nodes.")

In [ ]:
from shapely.geometry import LineString
import geopandas as gpd

greater_coords = set()
lesser_coords = set()
same_coords = set()
greater_val = set()
lesser_val = set()
same_val = set()

all_coords = set(demand_map.keys()).union(set(demand_map_new.keys()))

for coord in all_coords:
    val_old = demand_map.get(coord, 0)
    val_new = demand_map_new.get(coord, 0)

    if val_old < val_new:
        greater_coords.add(coord)
        greater_val.add(val_new-val_old)
    elif val_old > val_new:
        lesser_coords.add(coord)
        lesser_val.add(val_old-val_new)
    else:
        same_coords.add(coord)
        same_val.add(val_new-val_old)
unique_segments_inc = set()
unique_segments_dec = set()
unique_segments_same = set()

for route in routes_data_new:
    coords = list(route['geometry'].coords)

    for i in range(len(coords) - 1):
        pt_u = coords[i]
        pt_v = coords[i + 1]

        segment_id = (pt_u, pt_v)

        if pt_u in greater_coords:
            unique_segments_inc.add(segment_id)
        elif pt_u in lesser_coords:
            unique_segments_dec.add(segment_id)
        else:
            unique_segments_same.add(segment_id)

lines_inc = [LineString([u, v]) for u, v in unique_segments_inc]
lines_dec = [LineString([u, v]) for u, v in unique_segments_dec]
lines_same = [LineString([u, v]) for u, v in unique_segments_same]

gdf_routes_inc = gpd.GeoDataFrame(geometry=lines_inc, crs="EPSG:4326")
gdf_routes_dec = gpd.GeoDataFrame(geometry=lines_dec, crs="EPSG:4326")
gdf_routes_same = gpd.GeoDataFrame(geometry=lines_same, crs="EPSG:4326")

print(f"Unique Increased segments to draw: {len(gdf_routes_inc)}")
print(f"Unique Decreased segments to draw: {len(gdf_routes_dec)}")
print(f"Unique Same segments to draw: {len(gdf_routes_same)}")

In [ ]:
print((len(gdf_routes_dec)+len(gdf_routes_inc) + len(gdf_routes_same)))
print(len(gdf_routes_dec)/(len(gdf_routes_dec)+len(gdf_routes_inc) + len(gdf_routes_same)))
print(len(gdf_routes_same)/(len(gdf_routes_dec)+len(gdf_routes_inc) + len(gdf_routes_same)))
print(max(greater_val))
print(max(lesser_val))
# Calculate and print the peak relief demand
peak_relief = max(lesser_val) if lesser_val else 0
print(f"Peak forward demand shift (highest relief): {peak_relief} paths")

In [ ]:
total_increase = 0
total_relief = 0

for coord in all_coords:
    val_old = demand_map.get(coord, 0)
    val_new = demand_map_new.get(coord, 0)

    if val_new > val_old:
        total_increase += (val_new - val_old)
    elif val_old > val_new:
        total_relief += (val_old - val_new)

print(f"Total Demand Increase (Red): {total_increase}")
print(f"Total Demand Relief (Green): {total_relief}")

In [ ]:
# Export the GeoDataFrames to a single GeoPackage file
gdf_routes_inc.to_file("traffic_shift.gpkg", layer='Increased_Demand', driver="GPKG")
gdf_routes_dec.to_file("traffic_shift.gpkg", layer='Decreased_Demand', driver="GPKG")
gdf_routes_same.to_file("traffic_shift.gpkg", layer='Same_Demand', driver="GPKG")

print("Export complete! Download traffic_shift.gpkg from your Colab files.")

In [ ]:
print(max(gdf_routes_inc['geometry'].length))